# Coffee Shop Tea Inventory Automation

This project simulates a small inventory management system for a coffee shop.

The script:
- extracts tea sales from POS PDF reports
- updates the stock automatically
- detects low or critical inventory levels
- exports the updated stock to CSV

Technologies used:
- Python
- Pandas
- pdfplumber

# 1. Import Libraries

In [3]:
import pandas as pd
import numpy as np
import pdfplumber

# 2. Load Initial Stock
#### Load the initial tea inventory from CSV.

In [6]:
stock_df = pd.read_csv("stock_ceaiuri.csv")
stock_df.head(5)

,Produs,Stoc,Prag_minim
0,Green apple,5,2
1,Chai Matcha,5,2
2,Nana Mint,5,2
3,Green Energy,5,2
4,Mango Passionfruit,5,2


# 3. Inventory and Color Status Logic

In [7]:
def stock_status(row):
    if row['Stoc'] <= 0:
        return "CRITIC"
    elif row['Stoc'] <= row['Prag_minim']:
        return "LOW"
    else:
        return "OK"

In [51]:
def color_status(val):
    if val == "CRITIC":
        return "background-color: #ff4d4d"   # roșu
    elif val == "LOW":
        return "background-color: #ffd24d"   # galben
    elif val == "OK":
        return "background-color: #85e085"   # verde

# 4. Sales Simulation

In [42]:
#tea_sold = "Green apple"
#quantity = 1

#stock_df.loc[stock_df["Produs"] == tea_sold, "Stoc"] -= quantity

In [41]:
stock_df.head(1)

,Produs,Stoc,Prag_minim
0,Green apple,5,2


#### It works!

# 5. Extract Sales From POS PDF

In [26]:
pdf_path = "Top produse vândute 07-03-2026.pdf"

all_lines = []
with pdfplumber.open(pdf_path) as pdf:
    
    for page in pdf.pages:
        text = page.extract_text()
        lines = text.split("\n")
        all_lines.extend(lines)

all_lines[:5]

['Generat cu Freya Restaurant BackOffice v.3.5.28.14 © Soft Tehnica 2007-2026',
 'Unitate: Soft-Tehnica 2.11 Top produse vândute',
 'C.I.F.: 16819215',
 'Registrul Comerțului: un j',
 '07-03-2026 07:00:00 - 07-03-2026 23:00:00']

#### Now, we extract the sold tea.

In [36]:
tea_sales = []
for line in all_lines:
    if '100 G Buc' in line:
        parts = line.split()
        
     # găsim poziția unde apare '100'
        idx_100 = parts.index('100')
     # numele produsului = cuvintele dintre 1 și '100'
        name = ' '.join(parts[1:idx_100])
        
    # cantitatea e după 'Buc'
        qty = parts[parts.index("Buc")+1]
        qty=int(float(qty))
        
        tea_sales.append((name,qty))
tea_sales

[('Stomach Elixir', 1),
 ('Bon Appetit', 1),
 ('Raspberry Queen', 1),
 ('Love Tea', 1),
 ('Fasting Tea', 1)]

In [34]:
sales_df = pd.DataFrame(tea_sales, columns=["Produs", "Vandut"])
sales_df

,Produs,Vandut
0,Stomach Elixir,1
1,Bon Appetit,1
2,Raspberry Queen,1
3,Love Tea,1
4,Fasting Tea,1


In [44]:
sales_df.sort_values("Vandut", ascending=False)

,Produs,Vandut
0,Stomach Elixir,1
1,Bon Appetit,1
2,Raspberry Queen,1
3,Love Tea,1
4,Fasting Tea,1


In [45]:
stock_df["Produs"] = stock_df["Produs"].str.strip().str.lower()
sales_df["Produs"] = sales_df["Produs"].str.strip().str.lower()

# 6. Merge Sales With Stock

In [46]:
merged = stock_df.merge(sales_df, on="Produs", how="left")
merged["Vandut"] = merged["Vandut"].fillna(0).astype(int)
merged

,Produs,Stoc,Prag_minim,Vandut
0,green apple,5,2,0
1,chai matcha,5,2,0
2,nana mint,5,2,0
3,green energy,5,2,0
4,mango passionfruit,5,2,0
5,spicy inspiration,5,2,0
6,ginger orange,5,2,0
7,jasmins herb basket,5,2,0
8,a breeze of lavender,5,2,0
9,boost & energy,5,2,0


# 7. Update Inventory

In [47]:
merged["Stoc"] = merged["Stoc"] - merged["Vandut"]
merged["Status"] = merged.apply(stock_status, axis=1)

In [49]:
merged.head(2)

,Produs,Stoc,Prag_minim,Vandut,Status
0,green apple,5,2,0,OK
1,chai matcha,5,2,0,OK


# 8. Inventory Visualization

In [52]:
merged.style.map(color_status, subset=["Status"])

,Produs,Stoc,Prag_minim,Vandut,Status
0,green apple,5,2,0,OK
1,chai matcha,5,2,0,OK
2,nana mint,5,2,0,OK
3,green energy,5,2,0,OK
4,mango passionfruit,5,2,0,OK
5,spicy inspiration,5,2,0,OK
6,ginger orange,5,2,0,OK
7,jasmins herb basket,5,2,0,OK
8,a breeze of lavender,5,2,0,OK
9,boost & energy,5,2,0,OK


In [53]:
merged.to_csv("stock_ceaiuri.csv", index=False)